##### Importamos librerias

In [2]:
# Tratamiento de datos
import pandas as pd

# Exploración de archivos
import sys
import os

# Funciones personalizadas
sys.path.append(os.path.abspath('../src'))
import sp_limpieza as sl

# Visualizaciones
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Para mostrar todas las columnas y las filas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

##### Cargamos el conjunto de datos desde el archivo CSV

In [4]:
df_bank = pd.read_csv('../data/raw/bank-additional.csv', index_col=0)

In [5]:
df_bank.head()

,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_
0,NaN,housemaid,MARRIED,basic.4y,0.0,0.0,0.0,telephone,261,1,999,0,NONEXISTENT,1.1,"93,994","-36,4","4,857",5191,no,2-agosto-2019,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079
1,57.0,services,MARRIED,high.school,NaN,0.0,0.0,telephone,149,1,999,0,NONEXISTENT,1.1,"93,994","-36,4",NaN,5191,no,14-septiembre-2016,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097
2,37.0,services,MARRIED,high.school,0.0,1.0,0.0,telephone,226,1,999,0,NONEXISTENT,1.1,"93,994","-36,4","4,857",5191,no,15-febrero-2019,34.939,-94.847,3f9f49b5-e410-4948-bf6e-f9244f04918b
3,40.0,admin.,MARRIED,basic.6y,0.0,0.0,0.0,telephone,151,1,999,0,NONEXISTENT,1.1,"93,994","-36,4",NaN,5191,no,29-noviembre-2015,49.041,-70.308,9991fafb-4447-451a-8be2-b0df6098d13e
4,56.0,services,MARRIED,high.school,0.0,0.0,1.0,telephone,307,1,999,0,NONEXISTENT,1.1,"93,994","-36,4",NaN,5191,no,29-enero-2017,38.033,-104.463,eca60b76-70b6-4077-80ba-bc52e8ebb0eb


##### Limpieza de datos

In [6]:
# Eliminamos las colunmas 'latitude' y 'longitude'
df_bank = df_bank.drop(columns=['latitude', 'longitude'])

In [7]:
# Vamos a realizar la limpieza columna por columna en base al análisis preliminar.
df_bank.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'duration', 'campaign', 'pdays', 'previous', 'poutcome',
       'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m',
       'nr.employed', 'y', 'date', 'id_'],
      dtype='str')

##### Columna 'age'

In [8]:
# Columna 'age' tiene 5120 valores nulos (11.91 %). 
# Imputamos con la mediana para evitar la pérdida de información y reducir el impacto de valores extremos.
sl.imputar_mediana(df_bank, 'age')
print(f"Valores nulos restantes 'age': {df_bank['age'].isna().sum()}")

Valores nulos restantes 'age': 0


##### Columna 'job'

In [9]:
# Columna 'job' tiene 345 valores nulos (0.80 %). 
# Los imputamos como "unknown" para identificar clientes sin información laboral 
# sin alterar las categorías existentes.
sl.unknown(df_bank,'job')
print(f"Valores nulos restantes 'job': {df_bank['job'].isna().sum()}")
print(f"Valores 'unknown': {df_bank['job'].value_counts()['unknown']}")

Valores nulos restantes 'job': 0
Valores 'unknown': 345


##### Columna 'marital'

In [10]:
# Columna 'marital' tiene 85 valores nulos (0.20 %). Es una cantidad minúscula.
# Calculamos la moda de la columna y rellenamos los valores nulos con la moda calculada
sl.imputar_moda(df_bank, 'marital')
print(f"La moda para imputar es: {df_bank['marital'].mode()[0]}")
print(f"Valores nulos restantes en 'marital': {df_bank['marital'].isna().sum()}")

La moda para imputar es: MARRIED
Valores nulos restantes en 'marital': 0


In [11]:
# Cambiamos los nombres de las categorías a minúsculas
sl.lower(df_bank,'marital')
df_bank['marital'].unique()

<StringArray>
['married', 'single', 'divorced']
Length: 3, dtype: str

##### Columna 'education'

In [12]:
# Columna 'education' tiene 1807 valores nulos (4.20 %). 
# Los imputamos como "unknown" para conservar los registros y señalar la falta de información.
sl.unknown(df_bank, 'education')
df_bank['education'].unique()

<StringArray>
[           'basic.4y',         'high.school',            'basic.6y',
            'basic.9y', 'professional.course',             'unknown',
   'university.degree',          'illiterate']
Length: 8, dtype: str

##### Columna 'default'

In [13]:
# Columna 'default' tiene formato boolean. Tambien tiene 8981 valores nulos (20.89 %). 
# Comprobemos la distribución de los valores 0 y 1
df_bank['default'].value_counts()

default
0.0    34016
1.0        3
Name: count, dtype: int64

In [14]:
# Columna 'default' tiene casi el 99,99 % de sus valores iguales a 0.
# 34.016 frente a 3. Esta columna es inútil para el análisis porque genera "ruido". 
# La eliminamos
df_bank = df_bank.drop(columns=['default'])

##### Columna 'housing'

In [15]:
# Columna 'housing' tiene 1026 valores nulos (2.39 %).
# La distribución de los valores: 1.0 - 54% y 0.0 - 46%
# Los nulos imputamos con la categoría más frecuente (moda)
sl.imputar_moda(df_bank, 'housing')
print(f"Valores nulos restantes 'housing': {df_bank['housing'].isna().sum()}")

Valores nulos restantes 'housing': 0


In [16]:
# Transformamos la variable 'housing' de formato binario (0/1) a string (yes/no)
# para facilitar su interpretación en EDA
sl.binario_yes_no(df_bank, 'housing')
# Comprobamos los valores
df_bank['housing'].value_counts()

housing
yes    23524
no     19476
Name: count, dtype: int64

##### Columna 'loan'

In [17]:
# Columna 'loan' tiene 1026 valores nulos (2.39 %).
# La distribución de los valores: 1.0 - 16% y 0.0 - 84%
# Los nulos imputamos con la categoría más frecuente (moda)
sl.imputar_moda(df_bank, 'loan')
print(f"Valores nulos restantes 'loan': {df_bank['loan'].isna().sum()}")

Valores nulos restantes 'loan': 0


In [18]:
# Transformamos la variable 'loan' de formato binario (0/1) a string (yes/no)
sl.binario_yes_no(df_bank, 'loan')
# Comprobamos los valores
df_bank['loan'].value_counts()

loan
no     36468
yes     6532
Name: count, dtype: int64

##### Columnas 'contact', 'duration', 'compaign', 'pdays' dejamos como estan.

##### Columna 'poutcome'

In [19]:
# Cambiamos los nombres de las categorías a minúsculas
sl.lower(df_bank, 'poutcome')
# Comprobamos los valores:
df_bank['poutcome'].unique()

<StringArray>
['nonexistent', 'failure', 'success']
Length: 3, dtype: str

##### Columna 'emp.var.rate'. La dejamos sin cambios.

##### Columna 'cons.price.idx'

In [20]:
# Transformamos los valores string a valores numéricas.
# Primero reemplazamos la coma por el punto y después convertimos a float
sl.coma_float(df_bank, 'cons.price.idx')
print(df_bank['cons.price.idx'].head())

0    93.994
1    93.994
2    93.994
3    93.994
4    93.994
Name: cons.price.idx, dtype: float64


In [21]:
# Columna 'cons.price.idx' tiene 471 valores nulos (1.10 %).
# Imputamos con la mediana debido a su bajo porcentaje y naturaleza numérica.
sl.imputar_mediana(df_bank, 'cons.price.idx')
print(f"Mediana calculada: {df_bank['cons.price.idx'].median()}")
print(f"Nulos restantes: {df_bank['cons.price.idx'].isna().sum()}")

Mediana calculada: 93.749
Nulos restantes: 0


##### Columna 'cons.conf.idx'

In [22]:
# Transformamos los valores string a valores numéricas.
# Primero reemplazamos la coma por el punto y después convertimos a float
sl.coma_float(df_bank, 'cons.conf.idx')
df_bank['cons.conf.idx'].dtypes

dtype('float64')

##### Columna 'euribor3m'

In [23]:
# Transformamos los valores string a valores numéricas.
# Primero reemplazamos la coma por el punto y después convertimos a float
sl.coma_float(df_bank, 'euribor3m')
df_bank['euribor3m'].dtypes

dtype('float64')

In [24]:
# Columna 'euribor3m' tiene  9256 valores nulos (21.53 %).
# Tras ordenar por fecha, no existe relación entre los valores cero del 'euribor3m' y el periodo de tiempo.
# Pero hay una coherencia en que los datos van en orden.
# Utilizamos el método .interpolate(), que toma los valores anterior y siguiente y calcula su media.
df_bank['euribor3m'] = df_bank['euribor3m'].interpolate(method='linear').round(3)
df_bank['euribor3m'].isna().sum()

np.int64(0)

##### Columna 'nr.employed'

In [25]:
# Transformamos los valores string a valores numéricas.
# Primero reemplazamos la coma por el punto y después convertimos a float
sl.coma_float(df_bank, 'nr.employed')
df_bank['nr.employed'].dtypes

dtype('float64')

##### Columna 'y'

In [67]:
# Dado que la variable 'y' es un indicador clave de eficiencia de las campañas de marketing,
#  agregaremos una columna con sus valores numéricos para ampliar las capacidades de análisis.
df_bank['y_num'] = df_bank['y'].map({'no': 0, 'yes': 1})
# Aseguramos que ambas existen
print(df_bank[['y', 'y_num']].sample(5))

         y  y_num
2163    no      0
15477   no      0
15124   no      0
9103    no      0
13792  yes      1


##### Columna 'date'

In [27]:
# Convertimos la columna 'date' en formato de fecha.
# Primero creamos un diccionario de reemplazo meses por numeros.
# Pandas no reconoce nombres de meses en español.
meses_es = {
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04',
    'mayo': '05', 'junio': '06', 'julio': '07', 'agosto': '08',
    'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'}

# # Cambiamos los nombres de los meses a números dentro de la cadena
for mes_nom, mes_num in meses_es.items():
    df_bank['date'] = df_bank['date'].str.replace(mes_nom, mes_num, case=False)


In [28]:
# Convertimos la columna 'date' al tipo 'datetime'
df_bank['date'] = pd.to_datetime(df_bank['date'], dayfirst=True)
# Comprobamos tipo de dato de la columna 'date'
print(df_bank['date'].dtype)

datetime64[us]


In [29]:
# Columna 'date' tiene 248 valores nulos (0.58 %).
# El porcentaje < 1%, pérdida de datos insignificante. Imputar fechas sería poco realista
# Eliminamos las filas con fecha nula.
df_bank = df_bank.dropna(subset=['date'])
# Comprobamos el número de nulos
df_bank['date'].isna().sum()

np.int64(0)

##### Columnas 'contact_month' y 'contact_year'

In [30]:
# Estas columnas se encuentran en la descripción del conjunto de datos, pero no existen en él.
# Podrían ser útiles para el análisis. Vamos a agregarlos
df_bank['contact_month'] = df_bank['date'].dt.month
df_bank['contact_year'] = df_bank['date'].dt.year

##### Columna 'id_'

In [31]:
# Renombramos la columna 'id_' a 'ID' para unificar el identificador.
df_bank = df_bank.rename(columns={'id_': 'ID'})

##### Comprobamos como quedó el dataset limpio y transformado

In [68]:
df_bank.head()

,age,job,marital,education,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,ID,y_num,contact_month,contact_year
0,38.0,housemaid,married,basic.4y,no,no,telephone,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2019-08-02,089b39d8-e4d0-461b-87d4-814d71e0e079,0,8,2019
1,57.0,services,married,high.school,no,no,telephone,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2016-09-14,e9d37224-cb6f-4942-98d7-46672963d097,0,9,2016
2,37.0,services,married,high.school,yes,no,telephone,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2019-02-15,3f9f49b5-e410-4948-bf6e-f9244f04918b,0,2,2019
3,40.0,admin.,married,basic.6y,no,no,telephone,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2015-11-29,9991fafb-4447-451a-8be2-b0df6098d13e,0,11,2015
4,56.0,services,married,high.school,no,yes,telephone,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2017-01-29,eca60b76-70b6-4077-80ba-bc52e8ebb0eb,0,1,2017


##### Guardamos el dataset bancario df_bank limpio y transformado:

In [69]:
df_bank.to_csv('../data/processed/bank_limpio.csv', index=False)

##### Creamos y guardamos dataset final unido:

In [70]:
# Cargamos dataset de consumidores limpio para unirlo con el dataset de bank:
df_consum_limp = pd.read_csv('../data/processed/consum_limpio.csv', parse_dates=['Dt_Customer'])

In [71]:
# Unimos 2 conjuntos (bank y consumidores) por ID:
df_full = df_bank.merge(df_consum_limp, on='ID', how='left')

In [72]:
# Guardamos dataset unido y preparado para EDA
df_full.to_csv('../data/processed/unido_eda.csv', index=False)